# Autoencoder para detecção de spam em SMS

**Disciplina:** Redes Neurais  
**Integrantes:** preencher  
**Turma:** preencher

O sistema recebe uma mensagem SMS e indica se ela parece normal (`ham`) ou suspeita de spam. O autoencoder aprende a reconstruir mensagens legítimas; erros de reconstrução acima de um limite são tratados como anomalias.

> A base escolhida contém mensagens em inglês. O protótipo não foi validado para português.


## 1. Bibliotecas e carregamento dos dados

Usaremos a SMS Spam Collection, do UCI Machine Learning Repository (5.574 mensagens rotuladas como `ham` ou `spam`).

In [ ]:
%pip -q install tensorflow scikit-learn pandas matplotlib requests

import io, zipfile, time, random, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay, classification_report
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
response = requests.get(url, timeout=60); response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    filename = next(n for n in z.namelist() if n.endswith('SMSSpamCollection'))
    raw = z.read(filename).decode('utf-8', errors='replace')
rows = [line.split('\t', 1) for line in raw.splitlines() if '\t' in line]
df = pd.DataFrame(rows, columns=['label','text'])
df['label'] = df['label'].str.strip(); df['text'] = df['text'].str.strip()
df = df[df['label'].isin(['ham','spam']) & df['text'].ne('')].copy()
# Remove textos que aparecem com rótulos conflitantes e duplicatas.
ambiguous = df.groupby('text')['label'].nunique()
df = df[~df['text'].isin(ambiguous[ambiguous > 1].index)].drop_duplicates('text').reset_index(drop=True)
print('Mensagens após limpeza:', len(df))
display(df['label'].value_counts().rename(index={'ham':'Normal (ham)','spam':'Spam'}).to_frame('Quantidade'))

## 2. Divisão dos dados

Separação estratificada: 70% treino, 15% validação e 15% teste. O vocabulário TF-IDF será ajustado somente com o treino, para evitar vazamento de dados.

In [ ]:
X = df['text']; y = df['label']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f'Treino: {len(X_train)} ({len(X_train)/len(df):.1%})')
print(f'Validação: {len(X_val)} ({len(X_val)/len(df):.1%})')
print(f'Teste: {len(X_test)} ({len(X_test)/len(df):.1%})')
display(pd.DataFrame({'treino':y_train.value_counts(normalize=True),'validação':y_val.value_counts(normalize=True),'teste':y_test.value_counts(normalize=True)}))

## 3. Representação dos textos

TF-IDF converte cada mensagem em um vetor numérico: cada posição representa um termo ou combinação de termos e seu peso indica importância relativa. Não é um embedding contextual e não compreende plenamente o significado da frase.

In [ ]:
vectorizer = TfidfVectorizer(lowercase=True, strip_accents='unicode', max_features=1500, ngram_range=(1,2), min_df=2, max_df=0.98, norm='l2', dtype=np.float32)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)
print('Número de características:', X_train_tfidf.shape[1])
print('Exemplo de mensagem:', X_train.iloc[0])
print('Vetor TF-IDF (valores não zero):', X_train_tfidf[0].data[:10])

## 4. Baseline: regressão logística

A regressão logística é uma referência tradicional e supervisionada. Ela aprende diretamente com os rótulos `ham` e `spam`; `class_weight='balanced'` ajuda a compensar o desbalanceamento.

In [ ]:
baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
baseline.fit(X_train_tfidf, y_train)
pred_baseline = baseline.predict(X_test_tfidf)
score_baseline = baseline.predict_proba(X_test_tfidf)[:, list(baseline.classes_).index('spam')]
print('Baseline treinada.')

## 5. Arquitetura do autoencoder

O encoder comprime o vetor de entrada em um gargalo de 32 neurônios. O decoder tenta reconstruí-lo. O treinamento é **one-class/semi-supervisionado**, pois usamos os rótulos para selecionar apenas mensagens normais (`ham`) no treino. A rede não recebe a classe spam como alvo.

In [ ]:
Xtr = X_train_tfidf.toarray().astype(np.float32)
Xva = X_val_tfidf.toarray().astype(np.float32)
Xte = X_test_tfidf.toarray().astype(np.float32)
ytr = y_train.to_numpy(); yva = y_val.to_numpy(); yte = y_test.to_numpy()
Xtr_ham = Xtr[ytr == 'ham']; Xva_ham = Xva[yva == 'ham']
n_features = Xtr.shape[1]
inp = Input(shape=(n_features,), name='vetor_tfidf')
x = Dense(128, activation='relu')(inp)
x = Dense(64, activation='relu')(x)
latent = Dense(32, activation='relu', name='gargalo')(x)
x = Dense(64, activation='relu')(latent)
x = Dense(128, activation='relu')(x)
out = Dense(n_features, activation='sigmoid', name='reconstrucao')(x)
autoencoder = Model(inp, out)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
autoencoder.summary()

## 6. Treinamento e limite de anomalia

ReLU ativa as camadas ocultas; sigmoid mantém a saída de reconstrução entre 0 e 1. A perda MSE mede a diferença entre o vetor original e o reconstruído. Adam ajusta os pesos. Early Stopping reduz o risco de sobreajuste.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
start = time.time()
history = autoencoder.fit(Xtr_ham, Xtr_ham, validation_data=(Xva_ham, Xva_ham), epochs=40, batch_size=32, shuffle=True, callbacks=[early_stop], verbose=1)
training_seconds = time.time() - start
print(f'Épocas executadas: {len(history.history["loss"])} | Tempo: {training_seconds:.1f} s')
plt.figure(figsize=(8,4)); plt.plot(history.history['loss'], label='Treino'); plt.plot(history.history['val_loss'], label='Validação'); plt.xlabel('Época'); plt.ylabel('MSE'); plt.title('Perda de reconstrução'); plt.legend(); plt.grid(alpha=.2); plt.show()
def reconstruction_error(matrix):
    reconstructed = autoencoder.predict(matrix, verbose=0)
    return np.mean(np.square(matrix - reconstructed), axis=1)
val_errors_ham = reconstruction_error(Xva_ham)
threshold = float(np.percentile(val_errors_ham, 95))
print(f'Limite (percentil 95 do erro ham na validação): {threshold:.8f}')

## 7. Avaliação no conjunto de teste

Erro de reconstrução acima do limite é marcado como spam. As métricas de precisão, recall e F1 são calculadas para a classe spam. O conjunto de teste só é usado nesta etapa final.

In [ ]:
errors_test = reconstruction_error(Xte)
pred_ae = (errors_test > threshold).astype(int)
y_test_bin = (yte == 'spam').astype(int)
pred_base_bin = (pred_baseline == 'spam').astype(int)
def metrics(name, y_true, pred, scores):
    return {'Modelo':name, 'Acurácia':accuracy_score(y_true,pred), 'Precisão spam':precision_score(y_true,pred,zero_division=0), 'Recall spam':recall_score(y_true,pred,zero_division=0), 'F1 spam':f1_score(y_true,pred,zero_division=0), 'ROC-AUC':roc_auc_score(y_true,scores), 'Average Precision':average_precision_score(y_true,scores)}
results = pd.DataFrame([metrics('Regressão logística',y_test_bin,pred_base_bin,score_baseline), metrics('Autoencoder',y_test_bin,pred_ae,errors_test)]).set_index('Modelo')
display(results.style.format('{:.4f}'))
print('Relatório do autoencoder:')
print(classification_report(y_test_bin,pred_ae,labels=[0,1],target_names=['Normal','Spam'],zero_division=0))
fig, ax = plt.subplots(1,2,figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(y_test_bin,pred_base_bin,display_labels=['Normal','Spam'],ax=ax[0],colorbar=False); ax[0].set_title('Baseline')
ConfusionMatrixDisplay.from_predictions(y_test_bin,pred_ae,display_labels=['Normal','Spam'],ax=ax[1],colorbar=False); ax[1].set_title('Autoencoder')
plt.tight_layout(); plt.show()
print('Diferença F1 (autoencoder - baseline):', round(f1_score(y_test_bin,pred_ae)-f1_score(y_test_bin,pred_base_bin),4))

## 8. Análise de erros e demonstração

Falso positivo: mensagem legítima marcada como spam. Falso negativo: spam classificado como normal. Analise os exemplos abaixo para discutir causas possíveis.

In [ ]:
error_table = pd.DataFrame({'mensagem':X_test.to_numpy(),'esperado':yte,'previsto':np.where(pred_ae==1,'spam','ham'),'erro':errors_test})
print('Falsos positivos:')
display(error_table[(error_table.esperado=='ham') & (error_table.previsto=='spam')].head(3))
print('Falsos negativos:')
display(error_table[(error_table.esperado=='spam') & (error_table.previsto=='ham')].head(3))
def analisar_mensagem(texto):
    if not isinstance(texto,str) or not texto.strip(): raise ValueError('Digite uma mensagem não vazia.')
    vetor = vectorizer.transform([texto]).toarray().astype(np.float32)
    erro = float(reconstruction_error(vetor)[0])
    return {'mensagem':texto,'classificação':'SPAM (suspeita)' if erro > threshold else 'NORMAL (ham)','erro':erro,'limite':threshold,'observação':'O erro não é uma probabilidade de spam.'}
display(analisar_mensagem('Congratulations! You have won a free prize. Claim now!'))
nova_mensagem = input('Digite uma mensagem em inglês para analisar: ')
display(analisar_mensagem(nova_mensagem))

In [ ]:
# CÉLULA EXTRA — exemplos completos para o slide 13
# Execute depois da célula "Matrizes de confusão e análise dos erros".

# Evita que o Colab corte o texto das mensagens na tabela.
pd.set_option("display.max_colwidth", None)

# Exemplo de acerto: mensagem legítima (ham) prevista como ham.
acertos = analise_erros[
    (analise_erros["esperado"] == analise_erros["previsto_autoencoder"])
]

# Exemplos de erro: falsos negativos e falsos positivos já calculados acima.
print("EXEMPLO DE ACERTO — para o slide 13")
if not acertos.empty:
    exemplo_acerto = acertos.iloc[0]
    print("Mensagem:", exemplo_acerto["mensagem"])
    print("Esperado:", exemplo_acerto["esperado"])
    print("Previsto:", exemplo_acerto["previsto_autoencoder"])
else:
    print("Não foi encontrado exemplo de acerto nesta execução.")

print("\nEXEMPLO DE ERRO — falso negativo para o slide 13")
if not falsos_negativos.empty:
    exemplo_erro = falsos_negativos.iloc[0]
    print("Mensagem:", exemplo_erro["mensagem"])
    print("Esperado:", exemplo_erro["esperado"])
    print("Previsto:", exemplo_erro["previsto_autoencoder"])
    print("Erro de reconstrução:", f"{exemplo_erro['erro_reconstrucao']:.8f}")
else:
    print("Não foi encontrado falso negativo nesta execução.")

print("\nEXEMPLO DE ERRO — falso positivo (opcional)")
if not falsos_positivos.empty:
    exemplo_fp = falsos_positivos.iloc[0]
    print("Mensagem:", exemplo_fp["mensagem"])
    print("Esperado:", exemplo_fp["esperado"])
    print("Previsto:", exemplo_fp["previsto_autoencoder"])
else:
    print("Não foi encontrado falso positivo nesta execução.")


## 9. Limitações, ética e referências

- A base contém SMS em inglês e é histórica; não representa todo o spam atual nem mensagens em português.
- TF-IDF não compreende contexto ou intenção; mensagens legítimas incomuns podem gerar falsos positivos.
- O limite é uma heurística calibrada na validação, não uma garantia. O protótipo não deve bloquear mensagens automaticamente.
- Mensagens podem conter dados pessoais: minimizar coleta, proteger os dados e respeitar a legislação aplicável.

**Declaração de IA:** inteligência artificial generativa foi usada como apoio à organização, explicação dos conceitos e elaboração inicial do notebook. A equipe deve revisar, executar, adaptar e compreender o código.

**Referências:** UCI Machine Learning Repository, SMS Spam Collection (https://archive.ics.uci.edu/dataset/228/sms); Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR, 2011; Abadi et al., *TensorFlow: A System for Large-Scale Machine Learning*, 2016.